In [ ]:
# taken from https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html

#%matplotlib inline
import argparse
import os
import random
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.transforms.v2 as v2transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from PIL import Image
import skimage
from skimage import io, transform

In [ ]:
batch_size = 16
image_size = 324
model_image_size = 128
workers = 2
rootdir = "/Users/ishwar/Desktop"
output_dir = "model_output_020926_128x128_transformer"
if not os.path.isdir(output_dir):
    os.mkdir(output_dir)
NUM_LANDMARKS = 5

In [ ]:
class ChannelLayerNorm(nn.LayerNorm):
    def __init__(self, num_channels, channel_dim, num_dims):
        super(ChannelLayerNorm, self).__init__(num_channels)
        # build ro
        ro = []
        for i in range(num_dims-1):
            if i >= channel_dim:
                ro.append(i+1)
            else:
                ro.append(i)
        ro.append(channel_dim)

        ro_inv = []
        for i in range(num_dims):
            if i == channel_dim:
                ro_inv.append(num_dims-1)
            elif i > channel_dim:
                ro_inv.append(i-1)
            else:
                ro_inv.append(i)
        
        self.ro = tuple(ro)
        self.ro_inv = tuple(ro_inv)

        print(self.ro)
        print(self.ro_inv)

    def forward(self, x):
        x_perm = torch.permute(x, self.ro)
        layernorm_x = super().forward(x_perm)
        return torch.permute(layernorm_x, self.ro_inv)

In [ ]:
class ResidualNorm(nn.Module):
    def __init__(self, inchannels, outchannels):
        super(ResidualNorm, self).__init__()

        self.res_conv = nn.Conv2d(inchannels, outchannels, 1, stride=1, padding=0, bias=False)
        self.layer_norm = ChannelLayerNorm(outchannels, 1, 4)# nn.LayerNorm(outchannels)

    def forward(self, x_orig, x):
        res_image = x + self.res_conv(x_orig)
        # TODO: put layer norm before residual (+x)
        out_image = self.layer_norm(res_image) #torch.permute(self.layer_norm(torch.permute(res_image, (0, 2, 3, 1))), (0, 3, 1, 2))
        return out_image


In [ ]:
class Residual(nn.Module):
    def __init__(self, inchannels, outchannels):
        super(Residual, self).__init__()

        self.res_conv = nn.Conv2d(inchannels, outchannels, 1, stride=1, padding=0, bias=False)

    def forward(self, x_orig, x):
        res_image = x + self.res_conv(x_orig)
        return res_image

In [ ]:
class MLP(nn.Module):
    def __init__(self, inchannels, outchannels, num_layers=3):
        super(MLP, self).__init__()

        conv_layers = []
        for i in range(num_layers):
            endchannels = outchannels if (i == num_layers-1) else inchannels
            conv_layers.append(nn.Conv2d(inchannels, endchannels, 1, stride=1, padding=0, bias=False))
            conv_layers.append(nn.BatchNorm2d(endchannels))
            conv_layers.append(nn.LeakyReLU())

        self.conv_layers = nn.Sequential(*conv_layers)
        self.res = Residual(inchannels, outchannels)

    def forward(self, x):
        return self.res(x, self.conv_layers(x))


In [ ]:
class SameSizeConv(nn.Module):
    def __init__(self, inchannels, outchannels, p_dropout, num_layers=2):
        super(SameSizeConv, self).__init__()
        self.p_dropout = p_dropout
        
        self.kernelSize = 3
        convs = []

        for _ in range(num_layers):
            convs.append(nn.Conv2d(inchannels, inchannels, self.kernelSize, stride=1, \
                    padding=(self.kernelSize-1)//2, bias=False) )
            convs.append(nn.Conv2d(inchannels, inchannels, self.kernelSize+2, stride=1, \
                    dilation=2, padding=(self.kernelSize+2-1), bias=False) )
            convs.append(nn.BatchNorm2d(inchannels))
            convs.append(nn.LeakyReLU())
            convs.append(nn.Dropout(p=self.p_dropout))

        mlp = []
        mlp.append(nn.Conv2d(inchannels, inchannels, 1, stride=1, \
                            padding=0, bias=False) )
        mlp.append(nn.BatchNorm2d(inchannels))
        mlp.append(nn.LeakyReLU())
        mlp.append(nn.Dropout(p=self.p_dropout))

        mlp.append(nn.Conv2d(inchannels, outchannels, 1, stride=1, \
                            padding=0, bias=False) )
        mlp.append(nn.BatchNorm2d(outchannels))
        mlp.append(nn.LeakyReLU())
        mlp.append(nn.Dropout(p=self.p_dropout))

        self.res_conv = nn.Conv2d(inchannels, inchannels, 1, stride=1, padding=0, bias=False)
        self.res_mlp = nn.Conv2d(inchannels, outchannels, 1, stride=1, padding=0, bias=False)
        
        self.conv_layers = nn.Sequential(*convs)
        self.mlp_layers = nn.Sequential(*mlp)
    
    def forward(self, x):
        x2 = self.conv_layers(x) + self.res_conv(x)
        return self.mlp_layers(x2) + self.res_mlp(x2)

In [ ]:
class PatchConv(nn.Module):
    def __init__(self, in_channels, patch_size):
        super(PatchConv, self).__init__()
        with torch.no_grad():
            patch_arr = torch.reshape(torch.eye(patch_size**2), (patch_size**2, patch_size, patch_size))
            patch_weights = torch.zeros((in_channels * patch_size**2, in_channels, patch_size, patch_size))
            for i in range(in_channels * patch_size**2):
                patch_weights[i, i % in_channels] = patch_arr[i // in_channels]
            patch_weights.to(dtype=torch.float32)
            self.patch_filter = nn.Conv2d(in_channels, in_channels * patch_size**2, patch_size, stride=patch_size, padding=0, bias=False)
            self.patch_filter.weight = nn.Parameter(patch_weights)
        #self.patch_filter.weight.requires_grad = False
        # Freeze all parameters in the layer
        for param in self.patch_filter.parameters():
            param.requires_grad = False
    
    def forward(self, x):
        return self.patch_filter(x)

In [ ]:
# Absolute Position Encoding (32 channels)
def gen_ape(num_channels, img_size):
    num_channels = torch.tensor(num_channels) # must be a multiple of 4 to have sin & cos for both x and y axes
    img_size = torch.tensor(img_size)
    NQ = torch.tensor(4)

    varying_dim = torch.linspace(0, img_size-1, steps=img_size)
    static_dim = torch.linspace(0, 0, steps=img_size)
    wavelength_dim = torch.linspace(torch.log(NQ)/torch.log(img_size), 1, steps=num_channels//4)
    wavelength_dim = torch.pow(img_size, wavelength_dim)

    # make first dim position encoding
    grid_tuple = torch.meshgrid(wavelength_dim, varying_dim, static_dim, indexing='ij')
    dim1_ape = torch.concat(\
                        (torch.sin((1 /  grid_tuple[0]) * grid_tuple[1] * 2*torch.pi), 
                            torch.cos((1 /  grid_tuple[0]) * grid_tuple[1] * 2*torch.pi)) )
    # make second dim position encoding
    grid_tuple = torch.meshgrid(wavelength_dim, static_dim, varying_dim, indexing='ij')
    dim2_ape = torch.concat(\
                        (torch.sin((1 /  grid_tuple[0]) * grid_tuple[2] * 2*torch.pi), 
                            torch.cos((1 /  grid_tuple[0]) * grid_tuple[2] * 2*torch.pi)) )
    ape = torch.concat((dim1_ape, dim2_ape))

    #ape_seq = torch.permute(torch.flatten(ape, start_dim=1, end_dim=2), (1, 0))

    return ape

In [ ]:
def gen_learnable_ape(num_channels, img_size):
    x = torch.randn(num_channels, img_size, img_size)
    print(f"Learnable ape shape: {x.shape}")
    x_param = nn.Parameter(x, requires_grad=True)
    return x_param

In [ ]:
class VisionAttention(nn.Module):
    def __init__(self, image_size, inchannels, qkchannels, vchannels):
        super(VisionAttention, self).__init__()

        # need to know how to customize initialization broadly
        self.image_size = image_size
        self.qkchannels = qkchannels
        self.qkscalar = torch.sqrt(torch.tensor(self.qkchannels))
        
        self.vchannels = vchannels
        self.queryMatrix = nn.Parameter(torch.empty(inchannels, qkchannels), requires_grad=True)
        self.keyMatrix = nn.Parameter(torch.empty(inchannels, qkchannels), requires_grad=True)
        self.valueMatrix = nn.Parameter(torch.empty(inchannels, vchannels), requires_grad=True)

        self.ape = torch.permute(torch.flatten(gen_ape(qkchannels, self.image_size), start_dim=1, end_dim=2), (1, 0))

        self.res = ResidualNorm(inchannels, vchannels)

        self.mlp = MLP(vchannels, vchannels)
    
    def forward(self, x): # x should be of dimension (batch, channels, imx, imy) ->(batch, length of sequence, dimension of embedding)
        x_seq = torch.flatten(torch.permute(x, (0, 2, 3, 1)), start_dim=1, end_dim=2)
        queries = torch.matmul(x_seq+self.ape, self.queryMatrix)
        keys = torch.matmul(x_seq+self.ape, self.keyMatrix)
        values = torch.matmul(x_seq, self.valueMatrix)
        cosine_sim = torch.matmul(queries, torch.permute(keys, (0, 2, 1))) / self.qkscalar
        attention = F.softmax(cosine_sim, dim=2) # dimension 2 contains all dot products for one query and all keys
        values = torch.matmul(attention, values)
        values_image = torch.unflatten(torch.permute(values, (0, 2, 1)), dim=2, sizes=(self.image_size, self.image_size))

        out_image = self.res(x, values_image)

        mlp_image = self.mlp(out_image)

        return mlp_image

In [ ]:
SAVE_SWITCH=1
class MultiHeadVisionAttention(nn.Module):
    def __init__(self, image_size, inchannels, qkchannels, vchannels, numheads, given_ape=None):
        super(MultiHeadVisionAttention, self).__init__()

        # need to know how to customize initialization broadly
        self.image_size = image_size
        self.qkchannels = qkchannels
        self.qkscalar = torch.tensor(self.qkchannels)
        self.numheads = numheads

        self.initnorm = nn.LayerNorm(inchannels)
        
        self.vchannels = vchannels
        self.queryMatrix = nn.Parameter(torch.empty(numheads, inchannels, qkchannels), requires_grad=True)
        self.queryNorm = nn.LayerNorm(qkchannels)
        #self.register_parameter(name='queryMatrix', param=nn.Parameter(torch.empty(numheads, inchannels, qkchannels), requires_grad=True))
        self.keyMatrix = nn.Parameter(torch.empty(numheads, inchannels, qkchannels), requires_grad=True)
        self.keyNorm = nn.LayerNorm(qkchannels)
        #self.register_parameter(name='keyMatrix', param=nn.Parameter(torch.empty(numheads, inchannels, qkchannels), requires_grad=True))
        self.valueMatrix = nn.Parameter(torch.empty(numheads, inchannels, vchannels), requires_grad=True)
        #self.register_parameter(name='valueMatrix', param=nn.Parameter(torch.empty(numheads, inchannels, vchannels), requires_grad=True))
        self.valueNorm = ChannelLayerNorm(numheads*vchannels, 1, 4)

        # if given_ape == None:
        #     self.ape = gen_ape(qkchannels, self.image_size)
        # else:
        #     self.ape = given_ape # given_ape may or may not be learnable

        self.combine = nn.Conv2d(vchannels*numheads, vchannels, 1, stride=1, padding=0, bias=False)

        self.resnorm = ResidualNorm(inchannels, vchannels)

        self.mlp = MLP(vchannels, vchannels)
    
    def forward(self, x): # x should be of dimension (batch, channels, imx, imy) ->(batch, numheads, length of sequence, dimension of embedding)
        global SAVE_SWITCH
        x_seq = torch.permute(torch.flatten(x, start_dim=2, end_dim=3), (0, 2, 1))
        #x_seq_pos = x_seq + self.ape
        x_seq_heads = x_seq.reshape((x_seq.shape[0],)+(1,)+x_seq.shape[1:]).repeat(1, self.numheads, 1, 1)

        x_seq_heads = self.initnorm(x_seq_heads)

        #x_seq_pos_heads = x_seq_pos.reshape((x_seq.shape[0],)+(1,)+x_seq.shape[1:]).repeat(1, self.numheads, 1, 1)
        #print("Sequence shape after head dimension ", x_seq_heads.shape)
        queries = torch.matmul(x_seq_heads, self.queryMatrix)
        keys = torch.matmul(x_seq_heads, self.keyMatrix)
        values = torch.matmul(x_seq_heads, self.valueMatrix)

        ##queries_norm = self.queryNorm(queries)
        ##keys_norm = self.keyNorm(keys)

        ##cosine_sim = torch.matmul(queries_norm, torch.permute(keys_norm, (0, 1, 3, 2))) / self.qkscalar
        cosine_sim = torch.matmul(queries, torch.permute(keys, (0, 1, 3, 2))) / self.qkscalar

        #print(torch.max(torch.abs(cosine_sim)))

        attention = F.softmax(cosine_sim, dim=3) # dimension 3 contains all dot products for one query and all keys
        tsf_out = torch.matmul(attention, values)

        # tsf_out_image will now be of shape (batch, numheads, vchannels, image_size, image_size)
        tsf_out_image = torch.unflatten(torch.permute(tsf_out, (0, 1, 3, 2)), dim=3, sizes=(self.image_size, self.image_size))
        #print("Values image shape before head stack", values_image.shape)
        
        # tsf_out_image will now be of shape (batch, numheads*vchannels, image_size, image_size)
        tsf_out_image = torch.flatten(tsf_out_image, start_dim=1, end_dim=2)
        
        tsf_out_image_norm = self.valueNorm(tsf_out_image)

        #print("Values image shape after head stack", values_image.shape)
        combined = self.combine(tsf_out_image_norm)

        out_image = self.resnorm(x, combined)

        mlp_image = self.mlp(out_image)

        """
        if SAVE_SWITCH==1:
            torch.save(x_seq_heads, f"{output_dir}/x_seq_heads.pt")
            torch.save(queries, f"{output_dir}/queries.pt")
            torch.save(self.queryMatrix.data, f"{output_dir}/queryMatrix.pt")
            torch.save(keys, f"{output_dir}/keys.pt")
            torch.save(self.keyMatrix.data, f"{output_dir}/keyMatrix.pt")
            torch.save(values, f"{output_dir}/values.pt")
            torch.save(self.valueMatrix.data, f"{output_dir}/valueMatrix.pt")
            torch.save(cosine_sim, f"{output_dir}/cosine_sim.pt")
            torch.save(attention, f"{output_dir}/attention.pt")
            torch.save(tsf_out, f"{output_dir}/tsf_out.pt")
            torch.save(tsf_out_image, f"{output_dir}/tsf_out_image.pt")
            torch.save(out_image, f"{output_dir}/post_res_norm.pt")
            torch.save(mlp_image, f"{output_dir}/mlp_image.pt")
            SAVE_SWITCH=0
        """

        return mlp_image

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self, image_size, inchannels, outchannels, numheads, numlayers):
        super(VisionTransformer, self).__init__()
        
        self.intc = [256, 4] # 512 / 8 should be a perfect square (the patch size)
        self.patch_size = 8

        # self.conv_layers = []
        # self.conv_layers.append(nn.Conv2d(inchannels, 32, 1, stride=1, padding=0, bias=False))
        #self.conv_layers.append(SameSizeConv(32, 32, 0, num_layers=2))

        self.patch_conv = [PatchConv(inchannels, self.patch_size)] # image_size must be divisible by 8
        self.resizechannels = nn.Conv2d(inchannels*(self.patch_size**2), self.intc[0], 1, stride=1, padding=0, bias=False) # image_size must be divisible by 8
        self.norm1 = nn.BatchNorm2d(self.intc[0])

        self.transformer_layers = []
        
        self.learnable_ape = gen_learnable_ape(self.intc[0], image_size//self.patch_size)

        #self.transformer_layers.append(MultiHeadVisionAttention(image_size//self.patch_size, 4*(self.patch_size**2), 4*(self.patch_size**2), self.intc[0], numheads))
        ##self.transformer_layers.append(nn.ConvTranspose2d(self.intc[0], self.intc[1], self.patch_size, stride=self.patch_size, padding=0, bias=False) )
        ##self.transformer_layers.append(SameSizeConv(self.intc[1], self.intc[1], 0))

        for i in range(0, numlayers):
            ##self.transformer_layers.append(nn.Conv2d(self.intc[1], self.intc[0], self.patch_size, stride=self.patch_size, padding=0, bias=False))

            self.transformer_layers.append(MultiHeadVisionAttention(image_size//self.patch_size, self.intc[0], self.intc[0], self.intc[0], numheads))#, given_ape=self.learnable_ape))
            self.transformer_layers.append(nn.BatchNorm2d(self.intc[0]))
            #self.transformer_layers.append(nn.ConvTranspose2d(self.intc[0], self.intc[1], self.patch_size, stride=self.patch_size, padding=0, bias=False))
            #self.transformer_layers.append(SameSizeConv(self.intc[1], self.intc[1], 0))
        #self.transformer_layers.append( nn.ConvTranspose2d(self.intc[0], 16, 8, stride=8, padding=0, bias=False))
        self.transformer_layers.append( nn.ConvTranspose2d(self.intc[0], 64, 4, 2, 1, bias=False))
        self.transformer_layers.append(nn.BatchNorm2d(64))
        self.transformer_layers.append( nn.ConvTranspose2d(64, 16, 4, 2, 1, bias=False))
        self.transformer_layers.append(nn.BatchNorm2d(16))
        self.transformer_layers.append( nn.ConvTranspose2d(16, outchannels, 4, 2, 1, bias=False))
        self.transformer_layers.append(nn.BatchNorm2d(outchannels))

        # self.conv_layers = nn.Sequential(*self.conv_layers)
        self.transformer_layers = nn.Sequential(*self.transformer_layers)

    def forward(self, x):
        x2 = self.patch_conv[0](x)
        tsf_input = self.norm1(self.resizechannels(x2)) + self.learnable_ape
        return self.transformer_layers(tsf_input)


In [ ]:
def weights_init(m):
    #print(m)
    classname = m.__class__.__name__
    if isinstance(m, nn.Linear):
        print(f"Linear {m.weight.shape}")
        nn.init.normal_(m.weight.data, 0.0, (2/(m.weight.shape[1]))**0.5)
        if hasattr(m.bias, 'data'):
            nn.init.constant_(m.bias.data, 0.0)
    elif isinstance(m, MultiHeadVisionAttention):
        print(f"MultiHeadVisionAttention {m.queryMatrix.data.shape}")
        nn.init.normal_(m.queryMatrix.data, 0.0, (2/(m.queryMatrix.data.shape[1]))**0.5)
        nn.init.normal_(m.keyMatrix.data, 0.0, (2/(m.keyMatrix.data.shape[1]))**0.5)
        nn.init.normal_(m.valueMatrix.data, 0.0, (2/(m.valueMatrix.data.shape[1]))**0.5)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.05)
        nn.init.normal_(m.bias.data, 0.0, 0.001)
    elif isinstance(m, nn.Conv2d):
        print(f"Conv2d (2/({m.weight.shape[1]}*{m.weight.shape[2]}*{m.weight.shape[3]}))**0.5")
        nn.init.normal_(m.weight.data, 0.0, (2/(m.weight.shape[1]*m.weight.shape[2]*m.weight.shape[3]))**0.5) # shape[0] should be changed to shape[1]
        #print(m.weight.shape)
        if hasattr(m.bias, 'data'):
            nn.init.constant_(m.bias.data, 0.0)
    elif isinstance(m, nn.ConvTranspose2d):
        print(f"ConvTranspose2d (2/({m.weight.shape[0]}*(({m.weight.shape[2]}/{m.stride[0]})**2)))**0.5")
        nn.init.normal_(m.weight.data, 0.0, (2/(m.weight.shape[0]*((m.weight.shape[2]/m.stride[0])**2)))**0.5)
        #print(m.weight.shape)
        if hasattr(m.bias, 'data'):
            nn.init.constant_(m.bias.data, 0.0)
    m = m.to(torch.float32)

In [ ]:
# Set random seed for reproducibility
manualSeed = 499
#manualSeed = random.randint(1, 10000) # use if you want new results
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)

torch.use_deterministic_algorithms(True) # Needed for reproducible results

In [ ]:
#MIN_SIGMA = 1
MAX_SIGMA = 5
def gen_coordinates(coord_array):
    # make 7 coordinates from 5 coordinates
    coord_array_new = torch.zeros((NUM_LANDMARKS, 2))
    coord_array_new[0] = coord_array[0]
    coord_array_new[1] = coord_array[1]
    coord_array_new[2] = (1/2)*((3/5)*coord_array[3] + (2/5)*coord_array[0]) + (1/2)*coord_array[1] # on left wing
    coord_array_new[3] = coord_array[2]
    coord_array_new[4] = (1/2)*((3/5)*coord_array[3] + (2/5)*coord_array[0]) + (1/2)*coord_array[2] # on right wing
    coord_array_new[5] = coord_array[3]
    coord_array_new[6] = coord_array[4]

    coord_array_new[7] = (3/4)*coord_array[0] + (1/4)*coord_array[3]
    coord_array_new[8] = (1/2)*coord_array[0] + (1/2)*coord_array[3]
    coord_array_new[9] = (1/4)*coord_array[0] + (3/4)*coord_array[3]

    return coord_array_new

def coord_to_heatmap(coord_array, image_shape, SIGMA_RATIO, device):
    heatmaps = torch.zeros((NUM_LANDMARKS, image_shape[1], image_shape[2]))
    
    for i in range(NUM_LANDMARKS):
        mu_x, mu_y = coord_array[i][0], coord_array[i][1]   # center (x,y)
        #sigma = sigma_in * model_image_size / image_size
        # calculate average square root distance
        coord_diff = coord_array - torch.stack([coord_array[i],]*NUM_LANDMARKS)
        #print(coord_diff)
        mag = torch.linalg.vector_norm(coord_diff, 2, dim=1)
        mag_remove = torch.cat((mag[:i], mag[i+1:]))
        s = torch.pow(torch.mean(torch.pow(mag_remove + 1e-5, -1)), -1)
        #print(s)
        sigma = MAX_SIGMA #min(SIGMA_RATIO * s + MIN_SIGMA, MAX_SIGMA)

        ys = torch.arange(image_shape[1], device=device, dtype=torch.float32)
        xs = torch.arange(image_shape[2], device=device, dtype=torch.float32)
        yy, xx = torch.meshgrid(ys, xs, indexing='ij')          # HxW each

        img = torch.exp(-0.5 * ((xx - mu_x)**2 + (yy - mu_y)**2) / (sigma**2))
        #img = torch.sqrt((xx - mu_x)**2 + (yy - mu_y)**2)
        # (optional) normalize to peak (1 / 1 + 1e-3)
        #img = torch.round(img / (img.max() + 1e-5))
        heatmaps[i] = (img / (img.sum() + 1e-5))
        #heatmaps[i] = F.softmax(img.view((model_image_size*model_image_size,)), dim=0).view((model_image_size, model_image_size)) * (model_image_size*model_image_size)
    return heatmaps

In [ ]:
#SIGMA_RATIO = 0.05
class FramesFromCSV(Dataset):
    def __init__(self, csv_path, images_dir, image_pattern="frame_{frame}.png", transform=None, geo_transform=None, post_geo_transform=None):
        """
        Args:
            csv_path: path to CSV file with header in first row
            images_dir: directory containing frame PNGs
            image_pattern: naming pattern for image files
            transform: optional torchvision transform
        """
        self.images_dir = images_dir
        self.image_pattern = image_pattern
        self.transform = transform
        self.geo_transform = geo_transform
        self.post_geo_transform = post_geo_transform

        # Load CSV into numpy array, skipping header
        self.data = np.loadtxt(csv_path, delimiter=",", skiprows=1)
        # Ensure it's 2D even if only one row
        if self.data.ndim == 1:
            self.data = np.expand_dims(self.data, axis=0)

        self.frame_numbers = self.data[:, 0].astype(int)
        self.points = self.data[:, 1:]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        frame_num = self.frame_numbers[idx]
        coords = self.points[idx]

        # Load image
        img_path = os.path.join(self.images_dir, self.image_pattern.format(frame=frame_num))
        image = Image.open(img_path).convert("RGB")

        # # Reshape coords into N×2 array (x, y)
        coords = coords.reshape(-1, 2)
        coords = torch.tensor(coords, dtype=torch.float32)

        if self.transform:
            image = self.transform(image)
        if self.geo_transform:
            image, coords = self.geo_transform(image, coords)
        if self.post_geo_transform:
            image = self.post_geo_transform(image)

        #augmented_coords = gen_coordinates(coords)
        augmented_coords = coords * model_image_size / image_size
        normalized_image = image*2 - 1

        return {
            "image": normalized_image,
            "points": augmented_coords,
            "heatmaps": coord_to_heatmap(augmented_coords, image.shape, 0.1, "cpu"),
            "frame": frame_num
        }


In [ ]:
class RandomAdditiveNoise:
    def __init__(self, p=0.5, mean=0.0, std=0.05):
        self.p = p
        self.mean = mean
        self.std = std

    def __call__(self, x):
        # x: tensor in [0,1], shape (C, H, W)
        if torch.rand(1).item() < self.p:
            noise = torch.randn_like(x) * self.std + self.mean
            x = x + noise
            x = torch.clamp(x, 0.0, 1.0)
        return x

class RandomColorMatrix:
    """
    Applies x -> A x where A is 3x3 with:
      - rank in {1, 2, 3} (each w.p. 1/3)
      - spectral norm <= max_spectral_norm
    """
    def __init__(self, max_spectral_norm=1.5, min_spectral_norm=0.75):
        self.max_spectral_norm = max_spectral_norm
        self.min_spectral_norm = min_spectral_norm

    def _random_orthonormal(self):
        """Generate a random 3x3 orthonormal matrix via QR."""
        M = torch.randn(3, 3)
        Q, _ = torch.linalg.qr(M)  # Q is orthonormal
        return Q

    def __call__(self, x):
        # x: tensor in [0,1], shape (3, H, W)
        c, h, w = x.shape
        assert c == 3, "RandomColorMatrix expects a 3-channel RGB tensor."

        # # Choose rank 1, 2, or 3 with equal probability
        # rank = random.choice([1, 2, 3, 3, 3, 3, 3])

        # # Random orthonormals U, V
        # U = self._random_orthonormal()
        # V = self._random_orthonormal()

        # # Singular values: only 'rank' non-zero, each ≤ max_spectral_norm
        # s = torch.zeros(3)
        # nonzero_s = torch.rand(rank)*(self.max_spectral_norm - self.min_spectral_norm) + self.min_spectral_norm  # [min_spectral_norm, max_spectral_norm)
        # # Sort descending just for sanity (not required)
        # nonzero_s, _ = torch.sort(nonzero_s, descending=True)
        # s[:rank] = nonzero_s
        # S = torch.diag(s)

        # # A = U S V^T; spectral norm = max singular value <= max_spectral_norm
        # A = U @ S @ V.T  # shape (3,3)
        A = torch.eye(c)
        A += torch.randn_like(A)*0.1

        # Apply A to each pixel: x_flat is 3 x (H*W)
        x_flat = x.view(3, -1)
        x_transformed = A @ x_flat
        x_out = x_transformed.view(3, h, w)

        # Clamp back to [0,1]
        x_out = torch.clamp(torch.abs(x_out), 0.0, 1.0)
        return x_out

# random_affine = transforms.RandomAffine(
#     degrees=(-15, 15),           # rotation range in degrees
#     scale=(0.9, 1.1),            # scaling range
#     shear=(-10, 10),              # shear range in degrees
#     translate=(0.5, 0.5)
# )


In [ ]:
class ComposeJoint:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, img, pts):
        for t in self.transforms:
            img, pts = t(img, pts)
        return img, pts

# class ResizeWithPoints:
#     def __init__(self, size):
#         """
#         size: (H, W) or int (shorter side)
#         """
#         self.size = size

#     def __call__(self, img, pts):
#         # img.size -> (W, H)
#         old_w, old_h = img.size()

#         img = TF.resize(img, self.size())

#         if isinstance(self.size, tuple):
#             new_h, new_w = self.size()
#         else:
#             # torchvision's int size = shorter side; to keep it simple,
#             # assume you use (H, W) tuple for model_image_size
#             raise ValueError("Use an explicit (H, W) tuple for size")

#         sx = new_w / old_w
#         sy = new_h / old_h

#         scale = torch.tensor([sx, sy], dtype=torch.float32, device=pts.device)
#         pts = pts * scale  # elementwise [x, y] * [sx, sy]
#         return img, pts

class RandomAffineWithPoints:
    """
    Same params as torchvision.transforms.RandomAffine,
    but applied to both image and (N,2) coords.
    """
    def __init__(self, degrees, translate=None, scale=None, shear=None):
        self.degrees = degrees
        self.translate = translate
        self.scale = scale
        self.shear = shear

    def __call__(self, img, pts):
        # img.size is (W, H)
        w, h = img.size()[1:]

        # Sample random params exactly like RandomAffine
        angle, translations, scale, shear = transforms.RandomAffine.get_params(
            self.degrees, self.translate, self.scale, self.shear, (w, h)
        )
        corner_pixel = (img[0, -1, -1].item(), img[1, -1, -1].item(), img[2, -1, -1].item())
        # --- 1) Transform the image ---
        img = TF.affine(
            img,
            angle=angle,
            translate=translations,
            scale=scale,
            shear=shear,
            interpolation = transforms.InterpolationMode.BILINEAR,
            fill=corner_pixel
        )

        center = [w * 0.5, h * 0.5]
        inv_coeffs = TF._get_inverse_affine_matrix(
            center=center,
            angle=angle,
            translate=translations,
            scale=scale,
            shear=shear
        )
        # inv_coeffs = [a, b, c, d, e, f] giving:
        # x_in  = a * x_out + b * y_out + c
        # y_in  = d * x_out + e * y_out + f
        a, b, c, d, e, f = inv_coeffs
        #print(f"{a} {b} {c} {d} {e} {f}")

        M_inv = torch.tensor(
            [[a, b],
             [d, e]],
            dtype=torch.float32,
            device=pts.device,
        )

        M = torch.inverse(M_inv)  # 2×2

        # pts: (N,2) in input image coords
        #ones = torch.ones(pts.shape[0], 1, dtype=torch.float32, device=pts.device)
        #hom = torch.cat([pts, ones], dim=1).T  # 3×N

        #pts_out = (M @ hom).T[:, :2]  # back to (N,2)
        translation = torch.tensor([[c, f]]).repeat(pts.size()[0], 1)
        #print("Translation:\n", translation)
        #print("Points:\n", pts)

        pts_tr = pts - translation

        pts_out = (M @ pts_tr.T).T

        return img, pts_out
    
# geom_transform = ComposeJoint([
#     # ResizeWithPoints((model_image_size, model_image_size)),
#     RandomAffineWithPoints(
#         degrees=(-15, 15),
#         translate=(0.5, 0.5),
#         scale=(0.9, 1.1),
#         shear=(-10, 10),
#     ),
# ])

geom_transform = ComposeJoint([
    # ResizeWithPoints((model_image_size, model_image_size)),
    RandomAffineWithPoints(
        degrees=(-180, 180),
        translate=(0, 0),
        scale=(1.3, 1.5),
        shear=(-7, 7),
    ),
])

In [ ]:
dataset = FramesFromCSV(
    csv_path="plane_data/labeled_points_mega_grp.csv",
    images_dir="outp",
    image_pattern="frame_{frame}.png",
    transform=transforms.Compose([
        transforms.Resize(image_size),
        #random_affine,
        transforms.ToTensor(),
        ]),
    geo_transform=geom_transform,
    post_geo_transform=transforms.Compose([RandomColorMatrix(max_spectral_norm=1.5, min_spectral_norm=0.95),
                                           RandomAdditiveNoise(p=0.7, mean=0.0, std=0.025), transforms.Resize(model_image_size)])
)

In [ ]:
IMG_IDX = 291
IMG = dataset[IMG_IDX]
plt.imshow(torch.permute(IMG['image'], (1, 2, 0))*0.5 + 0.5)
# Unpack points into two lists: xs, ys
xs = [p[0] for p in IMG['points']]
ys = [p[1] for p in IMG['points']]
# Plot points as red circles
plt.scatter(xs, ys, c='red', s=10)  # s = point size

In [ ]:
print(torch.min(dataset[IMG_IDX]['image']))
print(torch.max(dataset[IMG_IDX]['image']))
print(dataset[IMG_IDX]['image'].shape)

In [ ]:
print(torch.min(IMG['heatmaps']), torch.max(IMG['heatmaps']))
print(IMG['heatmaps'].shape)
plt.imshow(IMG['heatmaps'][1])
plt.colorbar()

In [ ]:
print(torch.mean(IMG['heatmaps'][1]))

In [ ]:
print(IMG['points'])

In [ ]:
print(torch.min(IMG['heatmaps']), torch.max(IMG['heatmaps']))
plt.imshow(torch.sum(IMG['heatmaps'], axis=0))
plt.colorbar()

In [ ]:
total_size = len(dataset)
train_size = int(total_size * 0.8)
val_size = int(total_size * 0.1)
test_size = total_size - train_size - val_size # Ensure all samples are used

In [ ]:
seed = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], generator=seed)

print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")


In [ ]:
# loader = DataLoader(dataset, batch_size=batch_size, shuffle=True) # accidentally set shuffle to false all this time
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# for batch in train_loader:
#     print(batch["frame"])
#     print(batch["points"].shape)  # (B, N, 2)
#     break

In [ ]:
trial_number = 0

from datetime import datetime
import time

# instantiate generator, discriminator
# borrowed from https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html

device = torch.device("cuda:0" if (torch.cuda.is_available()) else "cpu")
print("device: ", device)

print("Instantiating VisionTransformer")
model = VisionTransformer(model_image_size, 3, NUM_LANDMARKS, 4, 3).to(device, dtype=torch.float32)
model.apply(weights_init)
print(model)

# Setup Adam optimizers for both G and D
optimizer = optim.Adam(model.parameters(), lr=0.0002, betas=(0.5, 0.999))
print("Model optimizer: \n", optimizer)

criterion = nn.MSELoss()

In [ ]:
print(model.patch_conv[0])
print(model.patch_conv[0].patch_filter.weight.requires_grad)
plt.hist(model.patch_conv[0].patch_filter.weight.flatten().detach().numpy(), bins=30)

In [ ]:
print(model.transformer_layers[8])
print(model.transformer_layers[8].weight)
plt.hist(model.transformer_layers[8].weight.flatten().detach().numpy(), bins=30)

In [ ]:
print(model.learnable_ape)
plt.hist(model.learnable_ape.flatten().detach().numpy(), bins=30)

In [ ]:
# Assuming 'model' is your defined torch.nn.Module instance
pytorch_total_params = sum(p.numel() for p in model.parameters())

pytorch_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {pytorch_total_params}")
print(f"Trainable parameters: {pytorch_trainable_params}")

In [ ]:
iters = 0
epoch = 0
num_epochs = 1000
#k=500 # saving images of fixed_noise outputs
last_save = iters
break_flag = False

losses = []

example_idx = 110

In [ ]:
def heatmap_softmax(heatmap_list, b, c):
    return [ F.softmax(heatmap.view(b, c, model_image_size*model_image_size), dim=2).view((b,c,model_image_size,model_image_size)) for heatmap in heatmap_list ]

In [ ]:
xs = torch.linspace(0.5, model_image_size-0.5, model_image_size, device=device) / model_image_size
ys = torch.linspace(0.5, model_image_size-0.5, model_image_size, device=device) / model_image_size
X, Y = torch.meshgrid(xs, ys, indexing="xy")
pts_template = torch.stack([X.reshape(-1), Y.reshape(-1)], dim=-1)  # (N, 2), N = grid_size^2

f_log = open(f"{output_dir}/training.log", "a")
f_log.write("""Checking if attention layers computations come out correctly; Cut most convolutional layers ... goal is to see whether attention only architecture works. Added batchnorms between final convolutional layers. Make ape learnable. Increased conv2d init std multiplier from 0.4 to 2. removed initial convolutional layer -- goes straight to patch conv and projection onto 256 dimensional embedding; only adding positional encoding at the start of the transformer layers by treating it as part of the input to the model (injected after conv layers and before transformer layers). More info on gradients of convolutional and MLP layers as well, changed patch size from 8 to 4; higher learning rate of 0.0012; added layernorm to normalize keys and queries before forming attention matrix (employing new ChannelLayerNorm class, which extends nn.LayerNorm to deal with the channel dim not being the last dim. Added a batchnorm after resizechannels convolution. Applied key and query transformations after normalizing x_seq input (for real this time, so that similarity matrix isn't just scaled by layernorm weights). Restored scaling factor of conv2d init to 1 (from 2). Scaled qk transformations init to 0.05 times original amount, left v transformation as is. Reduced learning rate to 0.0002. Changed attention scaling factor from sqrt(256) to 256.\n""")

example_indices = [110, 198]

print("Starting Training Loop...")
while (epoch < num_epochs) and (not break_flag):
    # For each batch in the dataloader
    should_break = False
    err_total_loss = 0

    f_log.write(f"Epoch\t{epoch}.\n")
    for i, data in enumerate(train_loader, 0):
    #while False:
        print(f"Epoch\t{epoch}, batch\t{i}.")
        t0 = time.perf_counter()

        data_input = data['image']
        output = data['heatmaps']
        output_coords = data['points'] / model_image_size

        data_input = data_input.to(device, dtype=torch.float32)

        b = data_input.shape[0]
        c = output.shape[1]

        t1 = time.perf_counter()
        f_log.write(f"Epoch\t{epoch}, batch\t{i}, {b} elements.\n")
        f_log.write(f"\t\tTime for setup: {t1-t0}\n")

        heatmap_logits = [model(data_input)]
        heatmap_probs = heatmap_softmax(heatmap_logits, b, c)
        predicted_heatmap_stages = heatmap_probs[0]##torch.cat(heatmap_predicted[0], dim=1)
        expected_output_stages = output #output.repeat(1, 3, 1, 1)

        t2 = time.perf_counter()
        f_log.write(f"\t\tTime for forward: {t2-t1}\n")

        model.zero_grad()

        t3 = time.perf_counter()
        f_log.write(f"\t\tTime for zero-grad: {t3-t2}\n")

        #print(expected_output_stages.shape)
        #print(predicted_heatmap_stages.shape)
        err_endpts_sum = criterion(predicted_heatmap_stages, expected_output_stages) * b / batch_size

        ###errs_endpts = []
        ###for j in range(len(heatmap_predicted[0])):
        ###    errs_endpts.append(criterion(heatmap_predicted[0][j], output))
        ###    print(f"Error for intermediate heatmap {j}: {errs_endpts[j].item()}")
        ###losses.append((epoch, i, iters, {f"Stage {j} error endpt": errs_endpts[j].item() for j in range(len(heatmap_predicted[0]))} ) )
        ###err_endpts_sum = (sum(errs_endpts)/len(errs_endpts)) * b / batch_size

        # errs_interior = []
        # for j in range(len(heatmap_predicted[1])):
        #     errs_interior.append(criterion(heatmap_predicted[1][j], output))
        #     print(f"Error for intermediate heatmap {j}: {errs_interior[j].item()}")
        
        # losses.append((epoch, i, iters, {f"Stage {j} error interior": errs_interior[j].item() for j in range(len(heatmap_predicted[1]))} ) )
        # err_interior_sum = (sum(errs_interior)/len(errs_interior)) * b / batch_size

        # err_sum = err_interior_sum*0.5 + err_endpts_sum*0.5
        err_sum = err_endpts_sum * (10**2)**3 # factor to adjust for the scale of the heatmaps
        losses.append( (epoch, i, iters, {f"Stage 0 error": err_sum.item()} ) )

        f_log.write(f"\tTraining loss / batch: {err_sum.item()}\n")
        err_total_loss += err_sum.item()

        t4 = time.perf_counter()
        f_log.write(f"\t\tTime for loss computation: {t4-t3}\n")

        err_sum.backward()

        t5 = time.perf_counter()
        f_log.write(f"\t\tTime for backprop: {t5-t4}\n")

        peek_modules = {"patch": model.patch_conv[0].patch_filter.weight, "init_resize": model.resizechannels.weight, \
                        "attn1q": model.transformer_layers[0].queryMatrix, "attn1k": model.transformer_layers[0].keyMatrix, "attn1v": model.transformer_layers[0].valueMatrix, \
                        "mlp1": model.transformer_layers[0].mlp,\
                        "attn2q": model.transformer_layers[2].queryMatrix, "attn2k": model.transformer_layers[2].keyMatrix, "attn2v": model.transformer_layers[2].valueMatrix, \
                        "mlp2": model.transformer_layers[2].mlp,\
                        "attn3q": model.transformer_layers[4].queryMatrix, "attn3k": model.transformer_layers[4].keyMatrix, "attn3v": model.transformer_layers[4].valueMatrix, \
                        "mlp3": model.transformer_layers[4].mlp,\
                        "up1": model.transformer_layers[6].weight, "up2": model.transformer_layers[6].weight, "up3": model.transformer_layers[6].weight,}
        f_log.write("\tGradient statistics\n")
        for num in peek_modules:
            all_gradients = []
            param = peek_modules[num]
            if isinstance(param, nn.Module):
                for name, param in peek_modules[num].named_parameters():
                    if param.grad is not None:
                        # Flatten the gradients and convert to numpy
                        all_gradients.append(param.grad.view(-1).numpy())
            elif isinstance(param, nn.Parameter):
                if param.grad is not None:
                    # Flatten the gradients and convert to numpy
                    all_gradients.append(param.grad.view(-1).numpy())
            # Concatenate all gradient arrays
            if len(all_gradients) == 0:
                f_log.write(f"\tPart {num} has no gradients to report.\n")
            else:
                all_gradients = np.abs(np.concatenate(all_gradients))
                f_log.write(f"\tPart {num}: {np.min(all_gradients):.2e} to {np.median(all_gradients):.2e} to {np.max(all_gradients):.2e}\n")
            #print(f"\tMedian. {np.median(all_gradients)}. Min {np.min(all_gradients)}. Max {np.max(all_gradients)}")
        f_log.write("\tValue statistics\n")
        for num in peek_modules:
            all_vals = []
            param = peek_modules[num]
            if isinstance(param, nn.Module):
                for name, param in peek_modules[num].named_parameters():
                    # Flatten the values and convert to numpy
                    all_vals.append(param.data.view(-1).numpy())
            elif isinstance(param, nn.Parameter):
                # Flatten the values and convert to numpy
                all_vals.append(param.data.view(-1).numpy())
            # Concatenate all values arrays
            all_vals = np.abs(np.concatenate(all_vals))
            f_log.write(f"\tPart {num}: {np.min(all_vals):.2e} to {np.median(all_vals):.2e} to {np.max(all_vals):.2e}\n")
            #print(f"\tMedian. {np.median(all_vals)}. Min {np.min(all_vals)}. Max {np.max(all_vals)}")

        t6 = time.perf_counter()
        f_log.write(f"\t\tTime for gathering statistics: {t6-t5}\n")
        
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1) # Clip to a max norm of 1.0

        t7 = time.perf_counter()
        f_log.write(f"\t\tTime for grad clip: {t7-t5}\n")

        optimizer.step()

        t8 = time.perf_counter()
        f_log.write(f"\t\tTime for opt step: {t8-t7}\n")

        iters += 1

        f_log.flush()

    f_log.write(f"Sum of losses over epochs (training): {err_total_loss}\n")
    f_log.write(f"Average of losses over epochs (training): {err_total_loss / train_size}\n")

    losses.append((epoch, -1, iters, {f"Average training loss": err_total_loss / train_size} ) )

    f_log.write(f"Epoch\t{epoch}. Evaluating validation loss\n")
    err_val_loss = 0
    model.eval()
    with torch.no_grad():
        for i, data in enumerate(val_loader, 0):
        #while False:
            t0 = time.perf_counter()

            data_input = data['image']
            output = data['heatmaps']
            output_coords = data['points'] / model_image_size

            data_input = data_input.to(device, dtype=torch.float32)

            b = data_input.shape[0]
            c = output.shape[1]

            t1 = time.perf_counter()
            f_log.write(f"Batch\t{i}, {b} elements.\n")
            f_log.write(f"\t\tTime for setup: {t1-t0}\n")

            heatmap_logits = [model(data_input)]
            heatmap_probs = heatmap_softmax(heatmap_logits, b, c)
            predicted_heatmap_stages = heatmap_probs[0]##torch.cat(heatmap_predicted[0], dim=1)
            expected_output_stages = output #output.repeat(1, 3, 1, 1)

            t2 = time.perf_counter()
            f_log.write(f"\t\tTime for forward: {t2-t1}\n")

            #print(expected_output_stages.shape)s
            #print(predicted_heatmap_stages.shape)
            err_endpts_sum = criterion(predicted_heatmap_stages, expected_output_stages) * b / batch_size

            ###errs_endpts = []
            ###for j in range(len(heatmap_predicted[0])):
            ###    errs_endpts.append(criterion(heatmap_predicted[0][j], output))
            ###    print(f"Error for intermediate heatmap {j}: {errs_endpts[j].item()}")
            ###losses.append((epoch, i, iters, {f"Stage {j} error endpt": errs_endpts[j].item() for j in range(len(heatmap_predicted[0]))} ) )
            ###err_endpts_sum = (sum(errs_endpts)/len(errs_endpts)) * b / batch_size

            # errs_interior = []
            # for j in range(len(heatmap_predicted[1])):
            #     errs_interior.append(criterion(heatmap_predicted[1][j], output))
            #     print(f"Error for intermediate heatmap {j}: {errs_interior[j].item()}")
            
            # losses.append((epoch, i, iters, {f"Stage {j} error interior": errs_interior[j].item() for j in range(len(heatmap_predicted[1]))} ) )
            # err_interior_sum = (sum(errs_interior)/len(errs_interior)) * b / batch_size

            # err_sum = err_interior_sum*0.5 + err_endpts_sum*0.5
            err_sum = err_endpts_sum * (10**2)**3 # factor to adjust for the scale of the heatmaps
            f_log.write(f"\tValidation loss / batch: {err_sum.item()}\n")
            err_val_loss += err_sum.item()

            t4 = time.perf_counter()
            f_log.write(f"\t\tTime for validation loss computation: {t4-t2}\n")

            f_log.flush()
    
    f_log.write(f"Total validation loss: {err_val_loss}\n")
    f_log.write(f"Average validation loss: {err_val_loss / val_size}\n")

    losses.append((epoch, -1, iters, {f"Average validation loss": err_val_loss / val_size} ) )
    print(f"Loss info length: {len(losses)}")

    model.train()
    
    if should_break:
        break

    if (epoch+1) % 1 == 0:
        for example_idx in example_indices:
            print(f"Saving results for example index {example_idx}")
            f_log.write(f"Saving results for example index {example_idx}\n")
            # output example output from model
            example_point = dataset[example_idx]
            image = torch.permute(example_point['image'], (1, 2, 0))
            plt.imshow(image.numpy())
            plt.savefig(f"{output_dir}/epoch{epoch}_i{example_idx}_in.png")
            plt.clf()

            input_example = example_point['image'].to(dtype=torch.float32)
            print(torch.min(input_example), torch.max(input_example))
            print(input_example.shape)

            model.eval()
            heatmap_pred = [model( torch.stack((input_example,)) )]
            heatmap_pred_detach = [y.detach() for y in heatmap_pred]
            heatmap_probs = heatmap_softmax(heatmap_pred, 1, c)
            heatmap_probs_detach = [y.detach() for y in heatmap_probs]
            model.train()

            heatmap_actual = example_point['heatmaps']
            print(heatmap_pred_detach[0].shape)
            for i in range(NUM_LANDMARKS):
                plt.imshow(heatmap_actual.numpy()[i])
                plt.colorbar()
                plt.savefig(f"{output_dir}/epoch{epoch}_i{example_idx}_out_{i}_real.png")
                plt.clf()
                for j in range(len(heatmap_pred_detach)):
                    plt.title(f"Heatmap {i} logits")
                    plt.imshow(heatmap_pred_detach[j].numpy()[0][i])
                    plt.colorbar()
                    plt.savefig(f"{output_dir}/epoch{epoch}_i{example_idx}_out_{i}_endpt{j}_logits.png")
                    plt.clf()
                    plt.title(f"Heatmap {i} probabilities")
                    plt.imshow(heatmap_probs_detach[j].numpy()[0][i])
                    plt.colorbar()
                    plt.savefig(f"{output_dir}/epoch{epoch}_i{example_idx}_out_{i}_endpt{j}_probs.png")
                    plt.clf()
                # for j in range(len(heatmap_pred_detach[1])):
                #     plt.imshow(heatmap_pred_detach[1][j].numpy()[0][i])
                #     plt.colorbar()
                #     plt.savefig(f"{output_dir}/epoch{epoch}_out_{i}_inter{j}.png")
                #     plt.clf()

    epoch += 1
    #SIGMA_RATIO = ((SIGMA_RATIO - 0.05) * 0.996) + 0.05
    #f_log.write(f"New SIGMA_RATIO: {SIGMA_RATIO}\n")

f_log.close()

In [ ]:
# new_beta2 = 0.99
# for param_group in optimizer.param_groups:
#     # param_group['betas'] is a tuple, so we create a new tuple
#     param_group['betas'] = (param_group['betas'][0], new_beta2)
new_lr = 0.0005
for param_group in optimizer.param_groups:
    param_group['lr'] = new_lr

In [ ]:
import math
validation_losses = [x[3]["Average validation loss"]*batch_size for x in losses if ((x[1] == -1) and ("Average validation loss" in x[3]))]
training_losses = [x[3]["Average training loss"]*batch_size for x in losses if ((x[1] == -1) and ("Average training loss" in x[3]))]
epoch_nums = [x[0] for x in losses if ((x[1] == -1) and ("Average training loss" in x[3]))]

batch_losses = [x[3]["Stage 0 error"] for x in losses if ((x[1] != -1) and ("Stage 0 error" in x[3]))]
iter_nums = [x[2] for x in losses if ((x[1] != -1) and ("Stage 0 error" in x[3]))]

In [ ]:
print(validation_losses[-5:])
print(training_losses[-5:])
print(epoch_nums[-5:])
print(batch_losses[-5:])
print(iter_nums[-5:])

In [ ]:
plt.title("Training loss vs Validation loss (softmax->MSE)")
plt.plot(epoch_nums, validation_losses, label="Validation loss")
plt.plot(epoch_nums, training_losses, label="Training loss")
plt.ylim(bottom=0)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.title("Log training loss vs log validation loss (softmax->MSE)")
plt.plot(epoch_nums, [math.log(x, 10) for x in validation_losses], label="Log validation loss")
plt.plot(epoch_nums, [math.log(x, 10) for x in training_losses], label="Log training loss")
plt.xlabel("Epochs")
plt.ylabel("Log Loss")
plt.legend()
plt.show()

In [ ]:
plt.title("Loss per gradient step (softmax->MSE)")
plt.plot(iter_nums, batch_losses, label="Loss / iteration")
plt.xlabel("Gradient steps")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.title("Log Loss per gradient step (softmax->MSE)")
plt.plot(iter_nums, [math.log(x, 10) for x in batch_losses], label="Log loss / iteration")
plt.xlabel("Gradient steps")
plt.ylabel("Log Loss")
plt.legend()
plt.show()

In [ ]:
torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': losses,
            }, f"{output_dir}/info_model_021026_872.pt")

In [ ]:
model_dict = torch.load(f"{output_dir}/info_model_021026_872.pt")
model_loaded = VisionTransformer(model_image_size, 3, NUM_LANDMARKS, 4, 3)
model_loaded.load_state_dict(model_dict['model_state_dict'])

In [ ]:
print(model_loaded)

Goal: search for where outputs across channels collapse to one result.

Initial conv: 
Patch conv: seemed that these weights changed, even after setting grad to false.
MHA1:
* query, key, value exhibit regular std (~0.1)
* mlp weights and residual linear seem half as much as what they should be -- sqrt(2/256) -- correct bc of 0.4 factor added originally
    * batchnorm2d weights range from 0.85 to 1.15, biases from -0.06 to 0.06
* residual: std around 0.08
* layer norm: 0.9 to 1.075 weight, -0.03 to 0.03 bias
* combine: sounds good
Found mistake in initialization scheme for Conv2D
MHA2: no abn
BatchNorm2D: no abn
MHA3: no abn
BatchNorm2D: no abns
ConvTranspose2D: need to work out initialization scheme here.
SameSizeConv: very last batchnorm has significant positive bias (0.2-0.7); otherwise, no abn

In [ ]:
examined_weights = model.patch_conv[0].patch_filter.weight

In [ ]:
print(examined_weights.shape)
plt.hist(examined_weights.flatten().detach().numpy(), bins=30)

In [ ]:
model_loaded.eval()

In [ ]:
#DATA_PT = 115
DATA_PT = 179
example_point = dataset[DATA_PT]
input_example = example_point['image'].to(dtype=torch.float32)

In [ ]:
image = torch.permute(example_point['image'], (1, 2, 0))
plt.imshow(image.numpy()*0.5 + 0.5)

In [ ]:
print(torch.min(input_example), torch.max(input_example))
print(input_example.shape)
SAVE_SWITCH=1
heatmap_pred = model( torch.stack((input_example,)) ).detach() # take unet outputted by first unet in the stacked hourglass
print(heatmap_pred.shape)
plt.imshow(heatmap_pred.numpy()[0][1])
plt.colorbar()

In [ ]:
# provided by Gemini

# 2. Create a dictionary to store activations
activations = {}

# 3. Define the hook function
def get_activation(name):
    def hook(model, input, output):
        # .detach() removes the tensor from the computation graph
        activations[name] = output.detach()
    return hook

# 4. Register hooks on desired layers
dims = {}

# 4. Register hooks on desired layers
model_loaded.transformer_layers[0].register_forward_hook(get_activation('transformer_layers[0]')); dims['transformer_layers[0]'] = [1, 256,16]
model_loaded.transformer_layers[2].register_forward_hook(get_activation('transformer_layers[2]')); dims['transformer_layers[2]'] = [1, 256,16]
model_loaded.transformer_layers[4].register_forward_hook(get_activation('transformer_layers[4]')); dims['transformer_layers[4]'] = [1, 256,16]
model_loaded.patch_conv[0].register_forward_hook(get_activation('patch_conv')); dims['patch_conv'] = [1,192, 16]
model_loaded.resizechannels.register_forward_hook(get_activation('resizechannels')); dims['resizechannels'] = [1,256, 16]
model_loaded.transformer_layers[6].register_forward_hook(get_activation('up1')); dims['up1'] = [1,64,32]
model_loaded.transformer_layers[8].register_forward_hook(get_activation('up2')); dims['up2'] = [1,16,64]
model_loaded.transformer_layers[10].register_forward_hook(get_activation('up3')); dims['up3'] = [1,5,128]

# model_loaded.unet1.upsize1.register_forward_hook(get_activation('unet1.upsize1')); dims['unet1.upsize1'] = [1, 16, 128]
# model_loaded.unet1.upsize2.register_forward_hook(get_activation('unet1.upsize2')); dims['unet1.upsize2'] = [2, 16, 64]
# model_loaded.unet1.upsize3.register_forward_hook(get_activation('unet1.upsize3')); dims['unet1.upsize3'] = [3, 32, 32]
# model_loaded.unet1.upsize4.register_forward_hook(get_activation('unet1.upsize4')); dims['unet1.upsize4'] = [4, 64, 16]
# model_loaded.unet1.convs0a.register_forward_hook(get_activation('unet1.convs0a')); dims['unet1.convs0a'] = [0, 16, 128]
# model_loaded.unet1.convs1a.register_forward_hook(get_activation('unet1.convs1a')); dims['unet1.convs1a'] = [1, 16, 64]
# model_loaded.unet1.convs2a.register_forward_hook(get_activation('unet1.convs2a')); dims['unet1.convs2a'] = [2, 32, 32]
# model_loaded.unet1.convs3a.register_forward_hook(get_activation('unet1.convs3a')); dims['unet1.convs3a'] = [3, 64, 16]
# model_loaded.unet1.lowest_convs.register_forward_hook(get_activation('unet1.lowest_convs')); dims['unet1.lowest_convs'] = [-1, 128, 8]
# model_loaded.unet1.final_conv.register_forward_hook(get_activation('unet1.final_conv')); dims['unet1.final_conv'] = [-1, 13, 128]

# 5. Perform a forward pass
#sample_input = torch.randn(1, 3, 32, 32) # Batch size 1, 3 channels, 32x32 image
_ = model_loaded( torch.stack((input_example,)) ) # Run the model

# 6. Access the stored activations
print(f"Shape of mha1 output: {activations['transformer_layers[0]'].shape}")

In [ ]:
def visualize_outputs(moniker, outputs):
    num_channels = outputs.shape[0]
    h = outputs.shape[1]
    w = outputs.shape[2]
    # Define the grid dimensions
    columns = 4
    rows = (num_channels+columns-1)//columns

    # find rank across output channels:
    output_mat = torch.flatten(outputs, start_dim=1).detach().numpy()
    print(output_mat.shape)
    s = np.linalg.svd(output_mat, compute_uv=False)
    # Select the top k singular values
    s_rank = [x for x in s if x > s[0]/100]
    print(f"\t{moniker}: Total Channels: {output_mat.shape[0]}, s_rank: {len(s_rank)}")

    # 2. Create figure and axes
    # fig is the entire figure, axs is a 2D array of axes (subplots)
    fig, axs = plt.subplots(rows, columns, figsize=(8, 3+rows))

    fig.suptitle(f"{moniker} features ({h}x{w}), srank={len(s_rank)}", fontsize=16, y=1)

    #images = [np.random.rand(10, 10) * i for i in range(1, 5)]
    vmin = torch.min(outputs).item()
    vmax = torch.max(outputs).item()
    # 2. Create the normalization and mappable
    norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = matplotlib.cm.ScalarMappable(norm=norm, cmap='viridis')

    titles = ["Channel "+str(i) for i in range(num_channels)]
    # 3. Iterate through axes and display images
    # Use .flatten() to turn the 2D array of axes into a 1D array for easy iteration
    for i, ax in enumerate(axs.flatten()):
        if i >= num_channels:
            break
        ax.imshow(outputs[i].numpy())
        ax.set_title(titles[i]) # Set a title for each subplot
        ax.axis('off') # Hide the axes ticks and labels for cleaner image presentation

    # Adjust layout for better spacing
    plt.tight_layout()
    fig.colorbar(sm, ax=axs, orientation='horizontal', label='Shared Scale')
    # Display the figure
    plt.savefig(f"{output_dir}/{moniker}_{DATA_PT}.png")

In [ ]:
for which_layer in dims:
    visualize_outputs(f"transformer_{which_layer}",  activations[which_layer][0])

In [ ]:
visualize_outputs(f"transformer_mha1_raw_out", torch.load(f"{output_dir}/tsf_out_image.pt").detach()[0])
visualize_outputs(f"transformer_mha1_combined", torch.load(f"{output_dir}/post_res_norm.pt").detach()[0])
visualize_outputs(f"transformer_mha1_mlp_out", torch.load(f"{output_dir}/mlp_image.pt").detach()[0])
visualize_outputs(f"transformer_mha1_values", torch.load(f"{output_dir}/values.pt").detach()[0])

In [ ]:
visualize_outputs(f"transformer_mha1_attentionmat", torch.load(f"{output_dir}/attention.pt").detach()[0])

In [ ]:
visualize_outputs(f"transformer_mha1_cosinesimmat", torch.load(f"{output_dir}/cosine_sim.pt").detach()[0])

In [ ]:
visualize_outputs(f"transformer_mha1_keys", torch.load(f"{output_dir}/keys.pt").detach()[0])

In [ ]:
visualize_outputs(f"transformer_mha1_queries", torch.load(f"{output_dir}/queries.pt").detach()[0])

In [ ]:
plt.hist(torch.load(f"{output_dir}/queryMatrix.pt").detach().numpy().flatten(), bins=30)

In [ ]:
plt.plot(range(256), np.linalg.svd(torch.load(f"{output_dir}/queryMatrix.pt").detach().numpy()[0], compute_uv=False))

In [ ]:
print(model_loaded.resizechannels)
print(model_loaded.resizechannels.weight.shape)
plt.hist(model_loaded.resizechannels.weight.detach().flatten(), bins=30)

In [ ]:
import frame_manager
import importlib
importlib.reload(frame_manager)

In [ ]:
SMOOTH_CONST = 1
# ---------- Main runner ----------
@torch.inference_mode()
def run_video_demo(model, source=0, device=None, window_name="Landmark Demo"):
    """
    source: 0 for webcam, or path to video file.
    model: a loaded torch.nn.Module that outputs heatmaps (B,C,Hh,Wh)
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)

    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video source: {source}")

    t_prev = time.time()
    fps_ema = None

    try:
        while True:
            ok, frame = cap.read()
            if not ok: break

            # Keep a copy for drawing at native res
            out_frame = frame.copy()

            # Preprocess & infer
            tin, orig_wh = preprocess_frame(frame, device)
            # Optional: mixed precision for speed on GPU
            with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                pred = F.softmax(model(tin), dim=1)                 # expect (1, C, Hh, Wh)
            if pred.dim() == 5: pred = pred[0]    # in case model returns (pred, aux)
            if pred.shape[0] == 1: pred = pred    # (1,C,Hh,Wh)
            _, C, Hh, Wh = pred.shape

            # Get coords in model-input space
            xs_in, ys_in = heatmaps_to_coords(pred)      # each (1,C)
            xs_in, ys_in = xs_in[0], ys_in[0]            # (C,)

            # Draw
            draw_points(out_frame, xs_in, ys_in, orig_wh)

            # FPS
            t_now = time.time()
            fps = 1.0 / max(t_now - t_prev, 1e-6)
            fps_ema = fps if fps_ema is None else 0.9*fps_ema + 0.1*fps
            t_prev = t_now
            put_fps(out_frame, fps_ema)

            cv2.imshow(window_name, out_frame)
            key = cv2.waitKey(1) & 0xFF
            if key in (27, ord('q')):  # ESC or q
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()


class LatestFrameGrabber:
    """Continuously grabs frames on a background thread and keeps only the most recent one."""
    def __init__(self, source=0):
        self.cap = cv2.VideoCapture(source)
        if not self.cap.isOpened():
            raise RuntimeError(f"Could not open video source: {source}")
        # Best-effort: keep tiny buffer (works on some backends; harmless otherwise)
        try:
            self.cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        except Exception:
            pass
        self.lock = threading.Lock()
        self.frame = None
        self.stopped = False
        self.t = threading.Thread(target=self._update, daemon=True)
        self.t.start()

    def _update(self):
        while not self.stopped:
            ok, f = self.cap.read()
            if not ok:
                self.stopped = True
                break
            with self.lock:
                self.frame = f

    def read(self):
        with self.lock:
            return None if self.frame is None else self.frame.copy()

    def release(self):
        self.stopped = True
        try:
            self.t.join(timeout=1.0)
        except Exception:
            pass
        self.cap.release()

import time, cv2, torch

@torch.inference_mode()
def run_video_demo_fixed_rate(model, source=0, device=None, window_name="Landmark Demo",
                              sample_hz=15.0, show_fps=True):
    """
    Samples the *latest* frame every 1/sample_hz seconds, runs inference on that frame,
    and displays only those results. Drops frames between ticks to avoid latency buildup.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)

    grab = LatestFrameGrabber(source)  # from previous code: grabs frames on a thread
    dt = 1.0 / float(max(sample_hz, 1e-6))
    next_t = time.perf_counter()  # monotonic clock

    disp_fps_ema = None
    try:
        while True:
            now = time.perf_counter()
            # sleep until the next scheduled tick (keeps rate steady, avoids drift)
            if now < next_t:
                time.sleep(next_t - now)
                continue
            # schedule the following tick
            next_t += dt

            t0 = time.perf_counter()
            frame = grab.read()
            if frame is None:
                # No new frame yet; skip this tick but keep schedule
                continue

            out_frame = frame.copy()

            # Preprocess & infer on the ticked frame
            tin, orig_wh = preprocess_frame(frame, device)
            with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                tin = tin.to(dtype=torch.float32)
                pred = model(tin)  # (1, C, Hh, Wh)

            xs_in, ys_in = heatmaps_to_coords(pred)   # each (1, C) in model-input coords
            xs_in, ys_in = xs_in[0], ys_in[0]
            draw_points(out_frame, xs_in, ys_in, orig_wh)

            # Display + UI FPS (UI FPS may be ~ sample_hz)
            t1 = time.perf_counter()
            ui_fps = 1.0 / max(t1 - t0, 1e-6)
            disp_fps_ema = ui_fps if disp_fps_ema is None else 0.9*disp_fps_ema + 0.1*ui_fps
            if show_fps:
                put_fps(out_frame, disp_fps_ema)
                cv2.putText(out_frame, f"Sample: {sample_hz:.1f} Hz",
                            (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2, cv2.LINE_AA)

            cv2.imshow(window_name, out_frame)
            key = cv2.waitKey(1) & 0xFF
            if key in (27, ord('q')):
                break

            # If inference was slower than dt, catch the schedule up (drop extra ticks)
            # so next tick isn't delayed by accumulated lag.
            now2 = time.perf_counter()
            while next_t < now2 - 1e-4:
                next_t += dt
    finally:
        grab.release()
        cv2.destroyAllWindows()

import time, cv2, torch

@torch.inference_mode()
def run_video_demo_auto(model, source=0, device=None, window_name="Landmark Demo",
                        sample_hz=15.0, show_fps=True):
    """
    - If source is webcam (fps not available) → run at fixed sample_hz (e.g. 15 Hz).
    - If source is video file (fps available) → throttle playback to the file's FPS.
    """

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)

    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video source: {source}")

    # Try to read FPS from file
    fps_file = cap.get(cv2.CAP_PROP_FPS)
    if fps_file is None or fps_file <= 1e-2:  # webcams often report 0
        mode = "webcam"
        dt = 1.0 / float(max(sample_hz, 1e-6))   # fixed rate
        print(f"[INFO] Webcam mode: target {sample_hz} Hz")
    else:
        mode = "file"
        dt = 1.0 / fps_file
        print(f"[INFO] Video file mode: target {fps_file:.2f} FPS")

    next_t = time.perf_counter()
    disp_fps_ema = None

    prev_xs_in_smooth, prev_ys_in_smooth = None, None

    prev_mask = [None] * 4
    prev_pred = None
    i = 0

    tau = 0.1

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            out_frame = frame.copy()

            i += 1
            if i % 5 != 0:
                continue

            # preprocess
            tin, orig_wh = frame_manager.preprocess_frame(frame, device)

            # inference
            with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                ##pred = F.softmax(model(tin), dim=1).detach()  # (1, C, H, W)
                model_output = model((tin*2 - 1))
                pred_logits = [x.detach() / tau for x in model_output]  # (1, C, H, W)
                pred_probs = heatmap_softmax(pred_logits, 1, NUM_LANDMARKS)
            """
            out_frame_2 = out_frame.copy().astype(np.float32)
            if (prev_pred is None):
                pred_smooth = pred
            else:
                pred_smooth = 0.3*pred + 0.7*prev_pred
            prev_pred = pred_smooth
            pred_round = torch.round(pred_smooth)
            for i in range(0, 3): 
                colored_heatmap = torch.permute(torch.tensor([255, 255, 255]) - torch.tensor(frame_manager.COLORS[i]).repeat(128, 128, 1), (2, 0, 1)) * pred_round[0][i].repeat(3, 1, 1).view(1, 3, 128, 128)
                # print(out_frame_2.shape)
                # print(colored_heatmap.shape)
                new_img = F.interpolate(colored_heatmap, size=(324, 324), mode='bilinear').to(torch.uint8)
                new_img_2 = (torch.ones(new_img.shape)*255 - new_img*0.5)/255
                # print(new_img.shape)
                mask = torch.permute(new_img_2, (0, 2, 3, 1)).view(324, 324, 3).numpy()
                
                out_frame_2 = out_frame_2 * mask

            out_frame_2 = out_frame_2.astype(np.uint8)
            """

            xs_in, ys_in = frame_manager.heatmaps_to_coords(pred_probs[0])
            #xs_in, ys_in = xs_in[0], ys_in[0]
            if prev_xs_in_smooth is None:
                xs_in_smooth, ys_in_smooth = xs_in, ys_in
            else:
                xs_in_smooth, ys_in_smooth = xs_in*SMOOTH_CONST + prev_xs_in_smooth*(1-SMOOTH_CONST), ys_in*SMOOTH_CONST + prev_ys_in_smooth*(1-SMOOTH_CONST)
            frame_manager.draw_points(out_frame, xs_in_smooth, ys_in_smooth, orig_wh)
            prev_xs_in_smooth, prev_ys_in_smooth = xs_in_smooth, ys_in_smooth

            # UI FPS
            now = time.perf_counter()
            ui_fps = 1.0 / max(now - next_t + dt, 1e-6)
            disp_fps_ema = ui_fps if disp_fps_ema is None else 0.9*disp_fps_ema + 0.1*ui_fps
            if show_fps:
                frame_manager.put_fps(out_frame, disp_fps_ema)

            ## cv2.imshow(window_name, out_frame_2)
            cv2.imshow(window_name, out_frame)
            key = cv2.waitKey(1) & 0xFF
            if key in (27, ord('q')):
                break

            # pacing
            if mode == "webcam":
                # sleep until next scheduled tickq
                now = time.perf_counter()
                if now < next_t:
                    time.sleep(next_t - now)
                next_t += dt
            else:  # file
                wait_ms = int(max(1, dt * 1000))
                if cv2.waitKey(wait_ms) & 0xFF in (27, ord('q')):
                    break

    finally:
        cap.release()
        cv2.destroyAllWindows()


In [ ]:
# or a file:
run_video_demo_auto(model, source="plane_2.mov", sample_hz=60.0)

In [ ]:
DATA_PT = 10
example_point = dataset[DATA_PT]
#example_point = dataset[115]
image = torch.permute(example_point['image'], (1, 2, 0))
plt.imshow(image.numpy()*0.5 + 0.5)

In [ ]:
input_example = example_point['image'].to(dtype=torch.float32)
print(torch.min(input_example), torch.max(input_example))
print(input_example.shape)
heatmap_pred = model( torch.stack((input_example,)) )[0][2].detach()
print(heatmap_pred.shape)

In [ ]:
plt.imshow(heatmap_pred[0].numpy()[1])
plt.colorbar()